# 03 — Train/Test Split

**Project:** Financial Fraud Detection System  
**Step:** 7.1 — Prepare the Data and Create a Proper Train/Test Split  
**Dataset:** `data/raw/synthetic_fraud_dataset1 (1).csv`

---

## Objective

This notebook prepares the financial fraud dataset for future model training while **preventing data leakage**.  
All preprocessing transformations (encoding, scaling) are fitted **exclusively on training data** and then applied to both training and test sets.

**Scope of this notebook:**
- Load and prepare the dataset (reproduce cleaning from Step 6)
- Define features (`X`) and target (`y`)
- Validate feature/target separation
- Create a stratified train/test split (80/20)
- Verify class distribution preservation
- Build the preprocessing pipeline (consistent with Step 6)
- Fit preprocessing **only on training data**
- Transform training and test data
- Validate the transformed data

**Not in scope:** Model training, evaluation, SMOTE, hyperparameter tuning, deployment.  
Model training is **intentionally not included** in this notebook.

---
## 1. Import Required Libraries

In [1]:
import os
import warnings

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)
warnings.filterwarnings("ignore", category=FutureWarning)

print("Libraries imported successfully.")

Libraries imported successfully.


---
## 2. Load and Prepare the Dataset

Load the raw CSV and reproduce the same cleaning/feature engineering from Step 6.  
This notebook runs **independently** — no variables from Notebook 2 are assumed to exist.

In [2]:
# Robust relative path from notebooks/ to the raw data file
DATA_PATH = os.path.join("..", "data", "raw", "synthetic_fraud_dataset1 (1).csv")

raw_df = pd.read_csv(DATA_PATH)
print(f"Raw dataset loaded from: {DATA_PATH}")
print(f"Shape: {raw_df.shape[0]:,} rows × {raw_df.shape[1]} columns")

# Create a working copy (raw CSV is never modified)
df = raw_df.copy()

# --- Reproduce Step 6 cleaning ---

# 1. Convert Date column from string to datetime
df["Date"] = pd.to_datetime(df["Date"], format="%d %B %Y", errors="coerce")
nat_count = df["Date"].isna().sum()
print(f"\nDate conversion: {nat_count} unparseable dates (NaT)")

# 2. Extract date-based features (consistent with Step 6)
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["Day"] = df["Date"].dt.day
df["Day_of_Week"] = df["Date"].dt.dayofweek  # Monday=0, Sunday=6

print("Date features created: Year, Month, Day, Day_of_Week")
print(f"Working DataFrame shape: {df.shape}")

Raw dataset loaded from: ..\data\raw\synthetic_fraud_dataset1 (1).csv
Shape: 50,000 rows × 14 columns

Date conversion: 0 unparseable dates (NaT)
Date features created: Year, Month, Day, Day_of_Week
Working DataFrame shape: (50000, 18)


---
## 3. Define Features and Target

- **Target (`y`):** `Fraud_Label`
- **Features (`X`):** All model input features, excluding identifiers (`Transaction_ID`, `User_ID`), the raw `Date` column, and the target.
- Date-derived features (`Year`, `Month`, `Day`, `Day_of_Week`) are included as numerical features.

In [3]:
# Columns to exclude from model features
exclude_from_X = ["Transaction_ID", "User_ID", "Date", "Fraud_Label"]

# Target
y = df["Fraud_Label"].copy()

# Features (everything except identifiers, raw Date, and target)
X = df.drop(columns=exclude_from_X).copy()

print(f"Target (y): shape = {y.shape}, dtype = {y.dtype}")
print(f"Features (X): shape = {X.shape}")
print(f"\nFeature columns ({X.shape[1]}):")
for i, col in enumerate(X.columns, 1):
    print(f"  {i:2d}. {col:35s} ({X[col].dtype})")

Target (y): shape = (50000,), dtype = int64
Features (X): shape = (50000, 14)

Feature columns (14):
   1. Transaction_Amount                  (float64)
   2. Transaction_Type                    (object)
   3. Account_Balance                     (float64)
   4. Device_Type                         (object)
   5. Location                            (object)
   6. Merchant_Category                   (object)
   7. Previous_Fraudulent_Activity        (int64)
   8. Daily_Transaction_Count             (int64)
   9. Card_Type                           (object)
  10. Card_Age                            (int64)
  11. Year                                (int32)
  12. Month                               (int32)
  13. Day                                 (int32)
  14. Day_of_Week                         (int32)


---
## 4. Validate Feature/Target Separation

Before splitting, verify that the feature matrix and target are properly separated with no leakage.

In [4]:
# --- Assertions ---
assert "Fraud_Label" not in X.columns, "ERROR: Fraud_Label found in X — target leakage!"
assert "Transaction_ID" not in X.columns, "ERROR: Transaction_ID found in X!"
assert "User_ID" not in X.columns, "ERROR: User_ID found in X!"
assert len(X) == len(y), "ERROR: X and y have mismatched row counts!"
assert y.isna().sum() == 0, "ERROR: Missing values in target!"
assert set(y.unique()).issubset({0, 1}), "ERROR: Target contains unexpected values!"

# Check for NaN/infinite values in numerical columns
X_numeric = X.select_dtypes(include=[np.number])
nan_count = X_numeric.isna().sum().sum()
inf_count = np.isinf(X_numeric).sum().sum()
assert nan_count == 0, f"ERROR: {nan_count} NaN values found in numerical features!"
assert inf_count == 0, f"ERROR: {inf_count} infinite values found in numerical features!"

print("Validation Results:")
print(f"  X rows: {len(X):,}  |  y rows: {len(y):,}  → Match: ✓")
print(f"  'Fraud_Label' in X: {('Fraud_Label' in X.columns)}  → ✓")
print(f"  'Transaction_ID' in X: {('Transaction_ID' in X.columns)}  → ✓")
print(f"  'User_ID' in X: {('User_ID' in X.columns)}  → ✓")
print(f"  Missing target values: {y.isna().sum()}  → ✓")
print(f"  Target unique values: {sorted(y.unique())}  → ✓")
print(f"  NaN in numerical features: {nan_count}  → ✓")
print(f"  Infinite in numerical features: {inf_count}  → ✓")
print("\n✓ All feature/target separation validations passed.")

Validation Results:
  X rows: 50,000  |  y rows: 50,000  → Match: ✓
  'Fraud_Label' in X: False  → ✓
  'Transaction_ID' in X: False  → ✓
  'User_ID' in X: False  → ✓
  Missing target values: 0  → ✓
  Target unique values: [np.int64(0), np.int64(1)]  → ✓
  NaN in numerical features: 0  → ✓
  Infinite in numerical features: 0  → ✓

✓ All feature/target separation validations passed.


---
## 5. Train/Test Split

Create a stratified 80/20 train/test split to preserve class distribution.  
Using `random_state=42` for reproducibility.

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Train/Test Split Complete")
print("=" * 50)
print(f"  Training set:  X_train {X_train.shape}  |  y_train {y_train.shape}")
print(f"  Testing set:   X_test  {X_test.shape}  |  y_test  {y_test.shape}")
print(f"\n  Total rows:    {len(X_train) + len(X_test):,}")
print(f"  Train rows:    {len(X_train):,}  ({len(X_train) / len(X) * 100:.1f}%)")
print(f"  Test rows:     {len(X_test):,}  ({len(X_test) / len(X) * 100:.1f}%)")
print(f"  random_state:  42")
print(f"  stratify:      y")

Train/Test Split Complete
  Training set:  X_train (40000, 14)  |  y_train (40000,)
  Testing set:   X_test  (10000, 14)  |  y_test  (10000,)

  Total rows:    50,000
  Train rows:    40,000  (80.0%)
  Test rows:     10,000  (20.0%)
  random_state:  42
  stratify:      y


---
## 6. Verify Class Distribution After Split

Confirm that stratification preserved the fraud/non-fraud ratio across all sets.  
No resampling (SMOTE, undersampling, oversampling) is applied — the natural distribution is retained.

In [6]:
def print_class_distribution(name, target_series, total_len):
    """Print class counts and percentages for a target series."""
    counts = target_series.value_counts().sort_index()
    print(f"\n  {name} (n={len(target_series):,}):")
    for val, count in counts.items():
        label = "Non-Fraud" if val == 0 else "Fraud"
        pct = count / len(target_series) * 100
        print(f"    {val} ({label}): {count:,}  ({pct:.2f}%)")

print("Class Distribution Verification")
print("=" * 50)

print_class_distribution("Complete Dataset", y, len(y))
print_class_distribution("Training Set", y_train, len(y_train))
print_class_distribution("Test Set", y_test, len(y_test))

# Verify stratification preserved ratios
full_ratio = y.value_counts(normalize=True).sort_index()
train_ratio = y_train.value_counts(normalize=True).sort_index()
test_ratio = y_test.value_counts(normalize=True).sort_index()

print("\n  Fraud Ratio Comparison:")
print(f"    Complete dataset: {full_ratio[1]:.4f}")
print(f"    Training set:     {train_ratio[1]:.4f}")
print(f"    Test set:         {test_ratio[1]:.4f}")
print("\n✓ Stratification verified — class distributions are consistent.")

Class Distribution Verification

  Complete Dataset (n=50,000):
    0 (Non-Fraud): 33,933  (67.87%)
    1 (Fraud): 16,067  (32.13%)

  Training Set (n=40,000):
    0 (Non-Fraud): 27,146  (67.86%)
    1 (Fraud): 12,854  (32.14%)

  Test Set (n=10,000):
    0 (Non-Fraud): 6,787  (67.87%)
    1 (Fraud): 3,213  (32.13%)

  Fraud Ratio Comparison:
    Complete dataset: 0.3213
    Training set:     0.3214
    Test set:         0.3213

✓ Stratification verified — class distributions are consistent.


---
## 7. Build Preprocessing Pipeline

Recreate the preprocessing design from Step 6 using `ColumnTransformer`.  

**Categorical features** → `OneHotEncoder(drop="first", handle_unknown="ignore")`  
**Numerical features** → `StandardScaler()`

The numerical features now include the 4 date-derived components (`Year`, `Month`, `Day`, `Day_of_Week`)  
in addition to the 5 original numerical columns, for a total of 9 numerical features.

In [7]:
# Define feature groups explicitly
categorical_features = [
    "Transaction_Type",
    "Device_Type",
    "Location",
    "Merchant_Category",
    "Card_Type",
]

numerical_features = [
    "Transaction_Amount",
    "Account_Balance",
    "Previous_Fraudulent_Activity",
    "Daily_Transaction_Count",
    "Card_Age",
    "Year",
    "Month",
    "Day",
    "Day_of_Week",
]

# Verify all specified columns exist in the DataFrame
for col in categorical_features + numerical_features:
    assert col in X_train.columns, f"ERROR: Column '{col}' not found in X_train!"

# Build ColumnTransformer (consistent with Step 6 design)
preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore"),
            categorical_features,
        ),
        (
            "num",
            StandardScaler(),
            numerical_features,
        ),
    ],
    remainder="drop",
)

print("Preprocessing ColumnTransformer defined:")
print(f"  Categorical → OneHotEncoder (drop='first', handle_unknown='ignore')")
print(f"    Columns ({len(categorical_features)}): {categorical_features}")
print(f"  Numerical  → StandardScaler")
print(f"    Columns ({len(numerical_features)}):  {numerical_features}")
print(f"\n  Total input features: {len(categorical_features) + len(numerical_features)}")

Preprocessing ColumnTransformer defined:
  Categorical → OneHotEncoder (drop='first', handle_unknown='ignore')
    Columns (5): ['Transaction_Type', 'Device_Type', 'Location', 'Merchant_Category', 'Card_Type']
  Numerical  → StandardScaler
    Columns (9):  ['Transaction_Amount', 'Account_Balance', 'Previous_Fraudulent_Activity', 'Daily_Transaction_Count', 'Card_Age', 'Year', 'Month', 'Day', 'Day_of_Week']

  Total input features: 14


---
## 8. Fit Preprocessing ONLY on Training Data

**This is the most critical step for preventing data leakage.**

The preprocessor is fitted **exclusively** on `X_train`.  
The test set (`X_test`) remains completely unseen during the fitting process.

- `preprocessor.fit(X_train)` — learns parameters from training data only
- **Never** `preprocessor.fit(X)` or `preprocessor.fit_transform(X)` before splitting

In [8]:
# Fit the preprocessor ONLY on training data
preprocessor.fit(X_train)

print("✓ Preprocessor fitted on X_train ONLY.")
print(f"  Training samples used for fitting: {len(X_train):,}")
print(f"  Test samples kept unseen: {len(X_test):,}")

✓ Preprocessor fitted on X_train ONLY.


  Training samples used for fitting: 40,000
  Test samples kept unseen: 10,000


---
## 9. Transform Training and Test Data

Apply the **same** fitted preprocessor to both training and test sets.  
No separate fitting is done on the test data.

In [9]:
# Transform both sets using the preprocessor fitted on X_train
X_train_processed = preprocessor.transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# Get feature names from the fitted preprocessor
transformed_feature_names = preprocessor.get_feature_names_out()

print("Transformation Complete")
print("=" * 50)
print(f"  X_train_processed shape: {X_train_processed.shape}")
print(f"  X_test_processed shape:  {X_test_processed.shape}")
print(f"\n  Transformed feature names ({len(transformed_feature_names)}):")
for i, name in enumerate(transformed_feature_names, 1):
    print(f"    {i:2d}. {name}")

Transformation Complete
  X_train_processed shape: (40000, 25)
  X_test_processed shape:  (10000, 25)

  Transformed feature names (25):
     1. cat__Transaction_Type_Bank Transfer
     2. cat__Transaction_Type_Online
     3. cat__Transaction_Type_POS
     4. cat__Device_Type_Mobile
     5. cat__Device_Type_Tablet
     6. cat__Location_Mumbai
     7. cat__Location_New York
     8. cat__Location_Sydney
     9. cat__Location_Tokyo
    10. cat__Merchant_Category_Electronics
    11. cat__Merchant_Category_Groceries
    12. cat__Merchant_Category_Restaurants
    13. cat__Merchant_Category_Travel
    14. cat__Card_Type_Discover
    15. cat__Card_Type_Mastercard
    16. cat__Card_Type_Visa
    17. num__Transaction_Amount
    18. num__Account_Balance
    19. num__Previous_Fraudulent_Activity
    20. num__Daily_Transaction_Count
    21. num__Card_Age
    22. num__Year
    23. num__Month
    24. num__Day
    25. num__Day_of_Week


---
## 10. Validate Transformed Data

Comprehensive validation of the transformed training and test datasets.

In [10]:
print("=" * 60)
print("TRANSFORMED DATA VALIDATION")
print("=" * 60)

# 1. Row count consistency
assert X_train_processed.shape[0] == X_train.shape[0], "Train row count mismatch!"
assert X_test_processed.shape[0] == X_test.shape[0], "Test row count mismatch!"
assert len(y_train) == X_train.shape[0], "y_train / X_train row mismatch!"
assert len(y_test) == X_test.shape[0], "y_test / X_test row mismatch!"
print("\n1. Row count consistency:")
print(f"   X_train_processed rows: {X_train_processed.shape[0]:,}  ==  X_train rows: {X_train.shape[0]:,}  → ✓")
print(f"   X_test_processed rows:  {X_test_processed.shape[0]:,}  ==  X_test rows:  {X_test.shape[0]:,}  → ✓")
print(f"   y_train rows: {len(y_train):,}  ==  X_train rows: {X_train.shape[0]:,}  → ✓")
print(f"   y_test rows:  {len(y_test):,}  ==  X_test rows:  {X_test.shape[0]:,}  → ✓")

# 2. Column count consistency between train and test
assert X_train_processed.shape[1] == X_test_processed.shape[1], "Train/test column count mismatch!"
print(f"\n2. Column count consistency:")
print(f"   Train features: {X_train_processed.shape[1]}  ==  Test features: {X_test_processed.shape[1]}  → ✓")

# 3. No NaN values
train_nan = np.isnan(X_train_processed).sum()
test_nan = np.isnan(X_test_processed).sum()
assert train_nan == 0, f"{train_nan} NaN values in processed training data!"
assert test_nan == 0, f"{test_nan} NaN values in processed test data!"
print(f"\n3. NaN values:")
print(f"   Training processed NaN count: {train_nan}  → ✓")
print(f"   Test processed NaN count:     {test_nan}  → ✓")

# 4. No infinite values
train_inf = np.isinf(X_train_processed).sum()
test_inf = np.isinf(X_test_processed).sum()
assert train_inf == 0, f"{train_inf} infinite values in processed training data!"
assert test_inf == 0, f"{test_inf} infinite values in processed test data!"
print(f"\n4. Infinite values:")
print(f"   Training processed inf count: {train_inf}  → ✓")
print(f"   Test processed inf count:     {test_inf}  → ✓")

# 5. Shape summary
print(f"\n5. Shape Summary:")
print(f"   Original training shape:   {X_train.shape}")
print(f"   Original testing shape:    {X_test.shape}")
print(f"   Processed training shape:  {X_train_processed.shape}")
print(f"   Processed testing shape:   {X_test_processed.shape}")

# 6. Confirm preprocessor was fitted on training data
print(f"\n6. Preprocessor fitting verification:")
print(f"   Preprocessor fitted: {hasattr(preprocessor, 'transformers_')}")
# The scaler should have mean_ from training data
num_transformer = preprocessor.named_transformers_["num"]
print(f"   StandardScaler has mean_: {hasattr(num_transformer, 'mean_')}")
print(f"   StandardScaler n_features: {num_transformer.n_features_in_}")
print(f"   StandardScaler n_samples_seen: {num_transformer.n_samples_seen_}")
print(f"   (Must equal training size {len(X_train):,})")
assert num_transformer.n_samples_seen_ == len(X_train), "Preprocessor was NOT fitted on X_train!"
print(f"   → ✓ Confirmed: preprocessor fitted on {num_transformer.n_samples_seen_:,} training samples only.")

print("\n" + "=" * 60)
print("✓ ALL TRANSFORMED DATA VALIDATIONS PASSED")
print("=" * 60)

TRANSFORMED DATA VALIDATION

1. Row count consistency:
   X_train_processed rows: 40,000  ==  X_train rows: 40,000  → ✓
   X_test_processed rows:  10,000  ==  X_test rows:  10,000  → ✓
   y_train rows: 40,000  ==  X_train rows: 40,000  → ✓
   y_test rows:  10,000  ==  X_test rows:  10,000  → ✓

2. Column count consistency:
   Train features: 25  ==  Test features: 25  → ✓

3. NaN values:
   Training processed NaN count: 0  → ✓
   Test processed NaN count:     0  → ✓

4. Infinite values:
   Training processed inf count: 0  → ✓
   Test processed inf count:     0  → ✓

5. Shape Summary:
   Original training shape:   (40000, 14)
   Original testing shape:    (10000, 14)
   Processed training shape:  (40000, 25)
   Processed testing shape:   (10000, 25)

6. Preprocessor fitting verification:
   Preprocessor fitted: True
   StandardScaler has mean_: True
   StandardScaler n_features: 9
   StandardScaler n_samples_seen: 40000
   (Must equal training size 40,000)
   → ✓ Confirmed: preprocessor

---
## 11. Final Train/Test Summary

In [11]:
# Compute fraud distributions for summary
train_fraud_counts = y_train.value_counts().sort_index()
test_fraud_counts = y_test.value_counts().sort_index()

print("=" * 60)
print("STEP 7.1 — FINAL TRAIN/TEST SUMMARY")
print("=" * 60)

print(f"\n  Original dataset size:        {len(X):,} rows")
print(f"  Training size:                 {len(X_train):,} rows")
print(f"  Testing size:                  {len(X_test):,} rows")
print(f"  Test size percentage:          {len(X_test) / len(X) * 100:.1f}%")
print(f"  Random state:                  42")
print(f"  Stratification:                Yes (stratify=y)")

print(f"\n  Training fraud distribution:")
print(f"    Non-Fraud (0): {train_fraud_counts[0]:,}  ({train_fraud_counts[0] / len(y_train) * 100:.2f}%)")
print(f"    Fraud (1):     {train_fraud_counts[1]:,}  ({train_fraud_counts[1] / len(y_train) * 100:.2f}%)")

print(f"\n  Testing fraud distribution:")
print(f"    Non-Fraud (0): {test_fraud_counts[0]:,}  ({test_fraud_counts[0] / len(y_test) * 100:.2f}%)")
print(f"    Fraud (1):     {test_fraud_counts[1]:,}  ({test_fraud_counts[1] / len(y_test) * 100:.2f}%)")

print(f"\n  Number of original model features:  {X_train.shape[1]}")
print(f"  Number of transformed features:      {X_train_processed.shape[1]}")
print("  Categorical encoding method:         OneHotEncoder(drop='first', handle_unknown='ignore')")
print("  Numerical scaling method:            StandardScaler()")

print(f"\n  ✓ Preprocessing fitted ONLY on X_train ({len(X_train):,} samples)")
print(f"  ✓ Test data was NEVER used to fit preprocessing")
print(f"  ✓ No data leakage")

print("\n" + "=" * 60)

STEP 7.1 — FINAL TRAIN/TEST SUMMARY

  Original dataset size:        50,000 rows
  Training size:                 40,000 rows
  Testing size:                  10,000 rows
  Test size percentage:          20.0%
  Random state:                  42
  Stratification:                Yes (stratify=y)

  Training fraud distribution:
    Non-Fraud (0): 27,146  (67.86%)
    Fraud (1):     12,854  (32.14%)

  Testing fraud distribution:
    Non-Fraud (0): 6,787  (67.87%)
    Fraud (1):     3,213  (32.13%)

  Number of original model features:  14
  Number of transformed features:      25
  Categorical encoding method:         OneHotEncoder(drop='first', handle_unknown='ignore')
  Numerical scaling method:            StandardScaler()

  ✓ Preprocessing fitted ONLY on X_train (40,000 samples)
  ✓ Test data was NEVER used to fit preprocessing
  ✓ No data leakage



---
## 12. Step 7.1 Completion

---

**Step 7.1 — Prepare the Data and Create a Proper Train/Test Split completed.**

Model training is intentionally not included. The project should proceed to Step 7.2 only after this step has been verified.